In [1]:
# data_generator.py
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ================= 配置区域 =================
# 路径配置
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

# 真实数据映射 (已更新)
FILE_MAPPING = {
    # 原有数据
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "QAM64",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "QAM16",
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "PSK8",
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    
    # 新增数据
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

# 扩充后的类别列表 (12类)
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 1. 数学与图像工具 =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_manifold_heatmap(iq_complex_data, img_size=64):
    """计算 R 通道 和 A 通道, Output: (2, H, W)"""
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    if iq_data.numel() < 100: return torch.zeros(2, img_size, img_size)

    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    real = norm_iq.real
    imag = norm_iq.imag
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    
    # R通道: 范围 [-1.2, 1.2]
    hist_r = fast_hist2d(real, imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    # A通道: 相位 [-1, 1], 幅度 [0, 1.5]
    hist_a = fast_hist2d(phase, mag, bins=img_size, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    
    def process(h):
        h = torch.log1p(h)
        return (h - h.min()) / (h.max() - h.min() + 1e-8)
        
    img_tensor = torch.stack([process(hist_r), process(hist_a)], dim=0) 
    return torch.flip(img_tensor, [1]) 

def save_tensor_as_dual_channel_png(tensor, path):
    """保存为 RGB PNG (R=Ch1, G=Ch2, B=0)"""
    tensor = tensor.detach().cpu()
    rgb_tensor = torch.zeros(3, tensor.shape[1], tensor.shape[2])
    rgb_tensor[0] = tensor[0] # Red = R Channel
    rgb_tensor[1] = tensor[1] # Green = A Channel
    arr = (rgb_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    Image.fromarray(arr, mode='RGB').save(path)

# ================= 2. 信号仿真核心 =================

def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    # 1. 载波频偏 (微偏)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    
    # 2. 初始相位随机
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    # 3. 相位噪声
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    
    # 4. IQ 不平衡
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    
    # 5. 高斯白噪声 (AWGN)
    sig_power = torch.mean(torch.abs(sig)**2)
    # 稍微提高一点信噪比基准，对齐真实数据质量
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

# ----- 星座图生成辅助函数 -----
def get_32qam_constellation():
    # 32QAM 十字形星座图
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            # 移除四个角的点
            if not ((abs(x) == 5 and abs(y) == 5) or (abs(x)==5 and abs(y)==3) or (abs(x)==3 and abs(y)==5)):
                points.append(x + 1j*y)
    # 补齐到32个点 (标准十字形通常是去掉角落)
    # 简化版：5x5=25 + 4x1? 不，32QAM标准是6x6去掉4个角(36-4=32)
    # 修正逻辑：
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            if not (abs(x) == 5 and abs(y) == 5): # 去掉最外角 (4个)
                points.append(x + 1j*y)
    return torch.tensor(points, dtype=torch.complex64)

def get_apsk_symbols(M, ratios, num_symbols):
    # APSK生成逻辑
    if M == 16:
        # 16APSK: R1(4点), R2(12点)
        R1, R2 = 1.0, ratios[0]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_outer = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        constellation = torch.cat([pts_inner, pts_outer])
    elif M == 32:
        # 32APSK: R1(4), R2(12), R3(16)
        R1, R2, R3 = 1.0, ratios[0], ratios[1]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_mid = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        pts_outer = R3 * torch.exp(1j * (torch.arange(16)*2*np.pi/16))
        constellation = torch.cat([pts_inner, pts_mid, pts_outer])
    
    indices = torch.randint(0, M, (num_symbols,))
    return constellation[indices]

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    syms = None
    
    # === 1. 符号生成 ===
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "8PSK":
        idx = torch.randint(0, 8, (num_symbols,))
        syms = torch.exp(1j * idx * 2*np.pi/8)
    elif mod_type == "OQPSK":
        # OQPSK: Q路延时半个符号周期
        idx = torch.randint(0, 4, (num_symbols,))
        temp_syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
        # 在上采样后处理OQPSK
        syms = temp_syms 
    elif mod_type == "PI4DQPSK":
        # pi/4 DQPSK: 差分相位调制
        bits = torch.randint(0, 4, (num_symbols,))
        # 相位跳变映射表: 00->pi/4, 01->3pi/4, 10->-pi/4, 11->-3pi/4
        phase_shifts = torch.tensor([np.pi/4, 3*np.pi/4, -np.pi/4, -3*np.pi/4])[bits]
        phases = torch.cumsum(phase_shifts, dim=0)
        syms = torch.exp(1j * phases)
    elif mod_type == "2ASK":
        # 2ASK: 幅度 0 或 1 (或者 -1, 1 但仅实部)
        # 这里使用 OOK 风格 (0, 1) 更容易在R通道区分
        bits = torch.randint(0, 2, (num_symbols,)).float()
        syms = bits + 0j
    elif mod_type == "16QAM":
        idx = torch.randint(0, 16, (num_symbols,))
        i = (idx // 4) * 2 - 3
        q = (idx % 4) * 2 - 3
        syms = i + 1j*q
    elif mod_type == "32QAM":
        constellation = get_32qam_constellation()
        idx = torch.randint(0, 32, (num_symbols,))
        syms = constellation[idx]
    elif mod_type == "64QAM":
        idx = torch.randint(0, 64, (num_symbols,))
        i = (idx // 8) * 2 - 7
        q = (idx % 8) * 2 - 7
        syms = i + 1j*q
    elif mod_type == "16APSK":
        # DVB-S2 标准半径比 R2/R1 约为 2.5 ~ 3.15
        syms = get_apsk_symbols(16, [2.7], num_symbols)
    elif mod_type == "32APSK":
        # DVB-S2 标准半径比 R2/R1 ~ 2.6, R3/R1 ~ 4.5
        syms = get_apsk_symbols(32, [2.6, 4.5], num_symbols)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return apply_channel_effects(sig, snr_db)

    # === 2. 上采样 & 成形滤波 ===
    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    
    if mod_type == "OQPSK":
        # OQPSK 特殊处理: Q 路延迟 sps/2 个点
        upsampled_i = torch.zeros(num_symbols * sps)
        upsampled_q = torch.zeros(num_symbols * sps)
        upsampled_i[::sps] = syms.real
        # Q路先上采样，再向后滚 sps//2
        q_temp = torch.zeros(num_symbols * sps)
        q_temp[::sps] = syms.imag
        upsampled_q = torch.roll(q_temp, shifts=sps//2)
        # 前面空的补0
        upsampled_q[:sps//2] = 0
        upsampled = upsampled_i + 1j * upsampled_q
    else:
        upsampled[::sps] = syms

    # RRC 滤波
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    
    return apply_channel_effects(sig, snr_db)

# ================= 3. 执行生成 =================

if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT):
        print("Cleaning old dataset...")
        shutil.rmtree(DATASET_ROOT)
    
    print("===== 1. Generating Simulated Data (R+A) for 12 Classes =====")
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        print(f"Generating {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            
            # 随机 SNR (-5 ~ 25dB)
            snr = np.random.uniform(-5, 25)
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            heatmap = generate_manifold_heatmap(sig.to(DEVICE), IMAGE_SIZE)
            
            # 保存
            save_path = os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.png")
            save_tensor_as_dual_channel_png(heatmap, save_path)

    print("\n===== 2. Processing Real Data (R+A) =====")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skip: {file_path}")
            continue
        print(f"Processing {label}...")
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q).astype(np.complex64) / 32767.0
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            for i in range(num_slices):
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                chunk_tensor = torch.from_numpy(chunk).to(DEVICE)
                heatmap = generate_manifold_heatmap(chunk_tensor, IMAGE_SIZE)
                save_tensor_as_dual_channel_png(heatmap, os.path.join(save_dir, f"real_{i:04d}_unknown.png"))
        except Exception as e:
            print(f"Error: {e}")
            
    print("\nData generation complete!")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cleaning old dataset...
===== 1. Generating Simulated Data (R+A) for 12 Classes =====
Generating BPSK...


100%|██████████| 800/800 [00:03<00:00, 224.00it/s]


Generating QPSK...


100%|██████████| 800/800 [00:02<00:00, 299.09it/s]


Generating 8PSK...


... (16 outputs left out)

100%|██████████| 800/800 [00:03<00:00, 227.86it/s]


Generating FSK2...


100%|██████████| 800/800 [00:02<00:00, 339.25it/s]



===== 2. Processing Real Data (R+A) =====
Processing BPSK...
Processing QAM64...
Processing QAM16...
Processing PSK8...
Processing QPSK...
Processing FSK2...
Processing 2ASK...
Processing OQPSK...
Processing PI4DQPSK...

Data generation complete!


In [2]:
# main_train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # 确保后台运行不弹窗
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results" 

# 12 类
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 15

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# ================= 1. 数据集定义 =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = [] 
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), 
            transforms.ToTensor()
        ])
        
        if not os.path.exists(root_dir):
            print(f"Path not found: {root_dir}")
            return
            
        print(f"Loading {root_dir} into RAM...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.png")])
        
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    snr = -999.0
                    if has_snr:
                        try:
                            fname = p.stem 
                            snr_str = fname.split('_')[-1]
                            snr = float(snr_str)
                        except: pass
                    
                    with Image.open(str(p)).convert("RGB") as img:
                        tensor = self.transform(img)
                        self.data.append(tensor[:2, :, :]) # 只取 R+A
                        self.labels.append(label)
                        self.snrs.append(snr)
                except Exception as e:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): 
        return self.data[idx], self.labels[idx], self.snrs[idx]

# ================= 2. 模型定义 =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 辅助绘图函数 =================
def plot_training_curves(history, save_path):
    epochs = range(1, len(history['train_acc']) + 1)
    plt.figure(figsize=(12, 5))
    
    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss') 
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ================= 4. 主训练逻辑 =================
def main():
    print("Initializing Datasets...")
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False) 
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True) 
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    if len(train_ds) == 0:
        print("No data found! Run data_generator.py first.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    best_val_acc = 0.0 # [新增] 记录最佳验证准确率
    
    print(f"\nStarting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels, _ in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        # --- Validation ---
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                val_running_loss += loss.item() * imgs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # [新增] 检查并保存最佳模型
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        
        log_msg = (f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                   f"Loss: {epoch_loss:.4f}/{val_loss:.4f} | "
                   f"Acc: {epoch_acc:.4f}/{val_acc:.4f}")
        
        if is_best:
            log_msg += "  [New Best! Saved]"
            
        print(log_msg)

    # 训练结束，保存曲线
    plot_training_curves(history, os.path.join(RESULTS_DIR, "training_curves.png"))
    
    # --- 5. 加载最佳模型进行评估 ---
    print("\nLoading Best Model for Final Evaluation...")
    if os.path.exists(BEST_MODEL_PATH):
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
    else:
        print("Warning: Best model not found, using last epoch weights.")

    model.eval()
    
    # --- Per-SNR Validation ---
    print("Evaluating Validation Set by SNR...")
    val_results = {'true': [], 'pred': [], 'snr': []}
    
    with torch.no_grad():
        for imgs, labels, snrs in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            
            val_results['true'].extend(labels.cpu().numpy())
            val_results['pred'].extend(preds.cpu().numpy())
            val_results['snr'].extend(snrs)
            
    y_true = np.array(val_results['true'])
    y_pred = np.array(val_results['pred'])
    snrs = np.array(val_results['snr'])
    
    snr_bins = [(-5, 0), (0, 5), (5, 10), (10, 15), (15, 20), (20, 30)]
    
    for (low, high) in snr_bins:
        mask = (snrs >= low) & (snrs < high)
        if np.sum(mask) == 0: continue
        
        subset_true = y_true[mask]
        subset_pred = y_pred[mask]
        subset_acc = np.mean(subset_true == subset_pred)
        
        title = f"SNR Range {low}dB to {high}dB (Acc: {subset_acc:.2%})"
        filename = os.path.join(RESULTS_DIR, f"cm_snr_{low}_{high}.png")
        
        plot_confusion_matrix(subset_true, subset_pred, title, filename)
        print(f"  -> Saved SNR {low}-{high}dB CM (Acc: {subset_acc:.2%})")

    # --- 6. 真实数据测试 ---
    if len(real_ds) > 0:
        print("\nRunning Real Data Test...")
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=False)
        real_true = []
        real_pred = []
        
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                real_true.extend(labels.cpu().numpy())
                real_pred.extend(preds.cpu().numpy())
        
        real_acc = np.mean(np.array(real_true) == np.array(real_pred))
        print(f"Real Data Accuracy: {real_acc:.4f}")
        
        plot_confusion_matrix(real_true, real_pred, 
                             f"Real Data Confusion Matrix (Acc: {real_acc:.2%})",
                             os.path.join(RESULTS_DIR, "cm_real_data.png"))
        print(f"Results saved to {RESULTS_DIR}")

if __name__ == "__main__":
    main()

Initializing Datasets...
Loading ./dataset_dual_RA\sim_train into RAM...


100%|██████████| 7723/7723 [00:29<00:00, 263.49img/s]


Loading ./dataset_dual_RA\sim_val into RAM...


100%|██████████| 1877/1877 [00:07<00:00, 263.41img/s]


Loading ./dataset_dual_RA\real_test into RAM...


100%|██████████| 29292/29292 [01:48<00:00, 270.00img/s]



Starting Training for 15 epochs...
Epoch 1/15 | Loss: 2.0014/1.2986 | Acc: 0.2891/0.6292  [New Best! Saved]
Epoch 2/15 | Loss: 1.2319/0.8353 | Acc: 0.5376/0.7214  [New Best! Saved]
Epoch 3/15 | Loss: 0.9705/0.6789 | Acc: 0.6355/0.7336  [New Best! Saved]
Epoch 4/15 | Loss: 0.8795/0.6396 | Acc: 0.6633/0.7560  [New Best! Saved]
Epoch 5/15 | Loss: 0.8232/0.5938 | Acc: 0.6817/0.7762  [New Best! Saved]
Epoch 6/15 | Loss: 0.7856/0.5955 | Acc: 0.6939/0.7693
Epoch 7/15 | Loss: 0.7799/0.5784 | Acc: 0.7037/0.7800  [New Best! Saved]
Epoch 8/15 | Loss: 0.7566/0.5835 | Acc: 0.7094/0.7880  [New Best! Saved]
Epoch 9/15 | Loss: 0.7397/0.5420 | Acc: 0.7128/0.7810
Epoch 10/15 | Loss: 0.7188/0.5631 | Acc: 0.7206/0.7901  [New Best! Saved]
Epoch 11/15 | Loss: 0.7102/0.5453 | Acc: 0.7210/0.7821
Epoch 12/15 | Loss: 0.7022/0.5228 | Acc: 0.7278/0.7965  [New Best! Saved]
Epoch 13/15 | Loss: 0.7011/0.5617 | Acc: 0.7272/0.7800
Epoch 14/15 | Loss: 0.6940/0.5286 | Acc: 0.7259/0.7954
Epoch 15/15 | Loss: 0.6832/0.560

In [3]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# ================= 配置 =================
# 必须与 main_train.py 中的完全一致
CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

REAL_TEST_DIR = "./dataset_dual_RA/real_test"

# ================= 诊断逻辑 =================
def diagnose():
    print(f"正在检查真实数据目录: {os.path.abspath(REAL_TEST_DIR)}")
    
    if not os.path.exists(REAL_TEST_DIR):
        print("❌ 错误：目录不存在！")
        return

    print("\n[步骤 1: 检查文件夹存在性]")
    print(f"{'期望类别':<15} | {'实际文件夹状态':<20} | {'图片数量':<10}")
    print("-" * 50)
    
    total_valid_images = 0
    
    for cls_name in CLASS_NAMES:
        cls_dir = os.path.join(REAL_TEST_DIR, cls_name)
        status = "✅ 存在" if os.path.exists(cls_dir) else "❌ 缺失 (名字不对?)"
        
        count = 0
        if os.path.exists(cls_dir):
            count = len(list(Path(cls_dir).glob("*.png")))
        
        print(f"{cls_name:<15} | {status:<20} | {count:<10}")
        
        if count > 0:
            # 抽查一张图片看能不能读
            try:
                img_path = next(Path(cls_dir).glob("*.png"))
                with Image.open(str(img_path)).convert("RGB") as img:
                    pass # 尝试读取，不报错就是好的
                total_valid_images += count
            except Exception as e:
                print(f"  ⚠️ 警告：{cls_name} 下的图片无法读取！错误：{e}")
                
    print("-" * 50)
    print(f"训练器预计能加载的图片总数: {total_valid_images}")
    
    # 额外检查：有没有不想干的文件夹？
    print("\n[步骤 2: 检查是否有命名错误的残留文件夹]")
    all_subdirs = [d for d in os.listdir(REAL_TEST_DIR) if os.path.isdir(os.path.join(REAL_TEST_DIR, d))]
    for d in all_subdirs:
        if d not in CLASS_NAMES:
            print(f"⚠️ 发现未知文件夹: '{d}' (不在 CLASS_NAMES 列表中，将被忽略)")
            print(f"   -> 建议：如果是你有数据的类别，请修改 data_generator.py 中的 FILE_MAPPING，将其重命名为标准名称。")

if __name__ == "__main__":
    diagnose()

正在检查真实数据目录: e:\Project\AMR-Benchmark\Other\dataset_dual_RA\real_test

[步骤 1: 检查文件夹存在性]
期望类别            | 实际文件夹状态              | 图片数量      
--------------------------------------------------
BPSK            | ✅ 存在                 | 4882      
QPSK            | ✅ 存在                 | 4882      
8PSK            | ❌ 缺失 (名字不对?)         | 0         
OQPSK           | ✅ 存在                 | 4882      
PI4DQPSK        | ✅ 存在                 | 4882      
16QAM           | ❌ 缺失 (名字不对?)         | 0         
32QAM           | ❌ 缺失 (名字不对?)         | 0         
64QAM           | ❌ 缺失 (名字不对?)         | 0         
2ASK            | ✅ 存在                 | 4882      
16APSK          | ❌ 缺失 (名字不对?)         | 0         
32APSK          | ❌ 缺失 (名字不对?)         | 0         
FSK2            | ✅ 存在                 | 4882      
--------------------------------------------------
训练器预计能加载的图片总数: 29292

[步骤 2: 检查是否有命名错误的残留文件夹]
⚠️ 发现未知文件夹: 'PSK8' (不在 CLASS_NAMES 列表中，将被忽略)
   -> 建议：如果是你有数据的类别，请修改 data_generator.py 中的